# Library Imports

The following libraries are imported to support data handling, visualization, image processing, model building, and training:

- **os, random**: For file system operations and randomization.
- **pandas, numpy**: For data manipulation and numerical operations.
- **matplotlib, seaborn**: For data visualization and plotting.
- **PIL (Pillow)**: For image processing.
- **OpenCV (cv2)**: For advanced image processing tasks.
- **TensorFlow and Keras modules**:
  - `ImageDataGenerator` for image augmentation.
  - Model-building layers such as `Conv2D`, `MaxPool2D`, `Dense`, `Flatten`, and `Dropout`.
  - Utilities like `to_categorical`.
- **scikit-learn**: For data splitting.
- **opendatasets**: For downloading datasets.
- `%matplotlib inline` magic command to display plots inline within Jupyter notebooks.

This setup ensures a comprehensive environment for image classification tasks using deep learning.


In [1]:
# Importing libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.image import imread
import seaborn as sns
import random
from PIL import Image
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.model_selection import  train_test_split
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout, Conv2D, MaxPool2D
import cv2
%matplotlib inline

2026-06-22 16:29:00.866250: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782145741.070936      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782145741.124409      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782145741.565181      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782145741.565232      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782145741.565236      58 computation_placer.cc:177] computation placer alr

# DownLoad Datasets

In [2]:
# od.download('https://www.kaggle.com/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign/')

In [ ]:

# Your classes dictionary
classes = {
    0:'Speed limit (20km/h)', 1:'Speed limit (30km/h)', 2:'Speed limit (50km/h)',
    3:'Speed limit (60km/h)', 4:'Speed limit (70km/h)', 5:'Speed limit (80km/h)',
    6:'End of speed limit (80km/h)', 7:'Speed limit (100km/h)', 8:'Speed limit (120km/h)',
    9:'No passing', 10:'No passing veh over 3.5 tons', 11:'Right-of-way at intersection',
    12:'Priority road', 13:'Yield', 14:'Stop', 15:'No vehicles', 16:'Veh > 3.5 tons prohibited',
    17:'No entry', 18:'General caution', 19:'Dangerous curve left', 20:'Dangerous curve right',
    21:'Double curve', 22:'Bumpy road', 23:'Slippery road', 24:'Road narrows on the right',
    25:'Road work', 26:'Traffic signals', 27:'Pedestrians', 28:'Children crossing',
    29:'Bicycles crossing', 30:'Beware of ice/snow', 31:'Wild animals crossing',
    32:'End speed + passing limits', 33:'Turn right ahead', 34:'Turn left ahead',
    35:'Ahead only', 36:'Go straight or right', 37:'Go straight or left', 38:'Keep right',
    39:'Keep left', 40:'Roundabout mandatory', 41:'End of no passing', 42:'End no passing veh > 3.5 tons'
}

train_image = "/kaggle/input/gtsrb-german-traffic-sign/Train/"

# Collect all image paths
image_paths = []
for root, dirs, files in os.walk(train_image):
    for file in files:
        if file.endswith(('.png', '.jpg', '.jpeg')):
            image_paths.append(os.path.join(root, file))

sample_paths = random.sample(image_paths, 25)

plt.figure(figsize=(15, 15))

for i, img_path in enumerate(sample_paths):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Extract label folder name (class id as string)
    label_str = os.path.basename(os.path.dirname(img_path))

    # Convert to int and get class name from dict
    label_int = int(label_str)
    class_name = classes.get(label_int, "Unknown")

    plt.subplot(5, 5, i+1)
    plt.imshow(img)
    plt.title(class_name, fontsize=7)
    plt.axis('off')

plt.tight_layout()
plt.show()


# Dataset Paths

- **Training Data Path:** `/content/gtsrb-german-traffic-sign/Train`
- **Testing Data Path:** `/content/gtsrb-german-traffic-sign/Test`

These directories contain the images used for training and testing the model respectively.


In [ ]:
train_path="/kaggle/input/gtsrb-german-traffic-sign/Train"

# Training Data Generator Setup

This configuration prepares the training data generator using Keras's `ImageDataGenerator` with data augmentation and validation splitting:

- **Batch Size:** 32
- **Random Seed:** 42 (for reproducibility)
- **Image Dimensions:** 50x50 pixels
- **Data Augmentation Techniques:**
  - Rescaling pixel values to [0, 1]
  - Random rotations up to 15 degrees
  - Horizontal and vertical shifts up to 10%
  - Shear transformations
  - Zoom in/out within 10%
  - Random horizontal flips
  - Filling empty pixels with nearest values after transformations
- **Validation Split:** 20% reserved for validation
- **Data Source:** Image paths and labels read from `Train.csv`
- **Label Format:** Converted to strings for compatibility with categorical class mode
- **Generator Behavior:** Shuffles data during training for better randomness
- **Subset:** Uses the `'training'` subset for training data, leaving the rest for validation

This setup efficiently loads and augments training images on the fly for robust model training.


In [ ]:
batch_size = 32
seed = 42
height=50
width=50
data_dir="/kaggle/input/gtsrb-german-traffic-sign"

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,          # Reserve 20% for validation
    rotation_range=15,             # Randomly rotate images by 0-15 degrees
    width_shift_range=0.1,         # Randomly shift images horizontally by 10%
    height_shift_range=0.1,        # Randomly shift images vertically by 10%
    shear_range=0.1,               # Shear transformations
    zoom_range=0.1,                # Random zoom in/out by 10%
    horizontal_flip=True,          # Randomly flip images horizontally
    fill_mode='nearest'            # Fill in new pixels after transformations
)

train_df = pd.read_csv(os.path.join(data_dir, 'Train.csv'))
train_df['ClassId'] = train_df['ClassId'].astype(str)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=data_dir,
    x_col='Path',
    y_col='ClassId',
    target_size=(height, width),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True,
    subset='training',
    seed=seed
)



# Visualizing a Batch of Training Images

The following code fetches one batch of images and labels from the training dataset generator and displays a 5x5 grid of 25 images with their corresponding class names as titles.

- **Images and labels** are retrieved from the generator using `next()`.
- **One-hot encoded labels** are converted to class indices with `np.argmax`.
- Class names are fetched from the `classes` dictionary.
- Images are plotted without axes for a clean visualization.


In [ ]:
# Get one batch of images and labels
images, labels = next(train_generator)

plt.figure(figsize=(12, 12))

for i in range(25):
    ax = plt.subplot(5, 5, i + 1)
    plt.imshow(images[i])
    # Get class index from one-hot encoded label and map to class name
    class_idx = np.argmax(labels[i])
    plt.title(classes[class_idx], fontsize=8)
    plt.axis('off')

plt.tight_layout()
plt.show()

# Test Data Generator Setup

The test dataset is prepared using Keras’s `ImageDataGenerator` with the following configuration:

- Test labels and image paths are loaded from a CSV file.
- Labels are converted to strings to comply with categorical mode requirements.
- Images are rescaled to have pixel values between 0 and 1.
- Images are loaded based on paths specified in the CSV relative to a base directory.
- Images are resized to the target dimensions `(height, width)`.
- Data is processed in batches defined by `batch_size`.
- Categorical class mode is used for multi-class classification.
- Data shuffling is disabled to maintain the order of samples for accurate evaluation.

This setup enables efficient and accurate feeding of test data to the model for evaluation or prediction.


In [ ]:
test_df = pd.read_csv(os.path.join(data_dir, 'Test.csv'))
test_df['ClassId'] = test_df['ClassId'].astype(str)

test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=data_dir,                  # Base directory for images
    x_col='Path',                       # Column with image paths relative to `directory`
    y_col='ClassId',                    # Column with labels
    target_size=(height, width),
    batch_size=batch_size,
    class_mode='categorical',
    suset="validation",
    shuffle=False                      # No shuffle for test generator
)

In [ ]:
train_generator.class_indices

# Class Distribution in Training Data

To understand the dataset balance, the number of images in each class folder within the training data is counted and compared.

This helps to:

- Identify any class imbalance.
- Determine if data augmentation or sampling strategies are needed.
- Guide model training decisions to ensure fair representation across classes.

By analyzing the class distribution, we can make informed choices for preprocessing and model evaluation.


In [ ]:
class_counts = {}

# Count images per class folder
for class_id in os.listdir(train_path):
    class_dir = os.path.join(train_path, class_id)
    if os.path.isdir(class_dir):
        count = len([f for f in os.listdir(class_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])
        class_counts[int(class_id)] = count

# Prepare DataFrame with class names and counts
data = {
    'Class ID': [],
    'Class Name': [],
    'Image Count': []
}

for class_id, count in sorted(class_counts.items()):
    data['Class ID'].append(class_id)
    data['Class Name'].append(classes.get(class_id, "Unknown"))
    data['Image Count'].append(count)

df = pd.DataFrame(data)

# Plotting
plt.figure(figsize=(20, 5))
sns.barplot(x='Class ID', y='Image Count', data=df,palette="Set1")
plt.xticks(ticks=df['Class ID'], labels=df['Class Name'], rotation=90)
plt.title('Number of Images per Class in Training Dataset')
plt.xlabel('Class Name')
plt.ylabel('Image Count')
plt.tight_layout()
plt.show()


# CNN Model Architecture Using `model.add()`

The convolutional neural network (CNN) model is built sequentially by adding layers one-by-one using `model.add()`:

- Starts with two convolutional layers (16 and 32 filters) with 5x5 kernels and ReLU activation.
- Applies max pooling and batch normalization to reduce spatial dimensions and stabilize training.
- Adds two more convolutional layers (64 filters each) with 3x3 kernels, followed by another max pooling, batch normalization, and dropout for regularization.
- Flattens the feature maps to a 1D vector.
- Adds a fully connected dense layer with 512 neurons and ReLU activation, batch normalization, and dropout.
- Ends with a dense output layer with 43 units (classes) and softmax activation for multi-class classification.

This layer-by-layer addition offers clear modularity and flexibility during model construction.


In [ ]:
from tensorflow import keras

model = keras.models.Sequential()

model.add(keras.layers.Conv2D(filters=16, kernel_size=(5,5), activation='relu', input_shape=(height, width, 3)))
model.add(keras.layers.Conv2D(filters=32, kernel_size=(5,5), activation='relu'))
model.add(keras.layers.MaxPool2D(pool_size=(2, 2)))
model.add(keras.layers.BatchNormalization(axis=-1))

model.add(keras.layers.Conv2D(filters=64, kernel_size=(3,3), activation='relu'))
model.add(keras.layers.Conv2D(filters=64, kernel_size=(3,3), activation='relu'))
model.add(keras.layers.MaxPool2D(pool_size=(2, 2)))
model.add(keras.layers.BatchNormalization(axis=-1))
model.add(keras.layers.Dropout(rate=0.25))

model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(512, activation='relu'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dropout(rate=0.25))

model.add(keras.layers.Dense(43, activation='softmax'))

optim = keras.optimizers.Adam(learning_rate=0.0001)
model.compile(optimizer = optim, loss = 'categorical_crossentropy', metrics = ['accuracy'])
model.summary()

# Plotting the Model Architecture

The model architecture is visualized and saved as an image using TensorFlow Keras's `plot_model` function with the following settings:

- The diagram is saved as `model_architecture.png`.
- Output shapes of each layer are displayed (`show_shapes=True`).
- Layer names are shown (`show_layer_names=True`).
- The image resolution is set with a DPI of 100 to control the size and clarity.



In [ ]:
from tensorflow.keras.utils import plot_model

plot_model(model, to_file='model_architecture.png', show_shapes=True, show_layer_names=True,dpi=100)


# Model Training

The model is trained using the `fit` method with the following parameters:

- **Training dataset:** `train_dataset`
- **Number of epochs:** 10
- **Verbose:** 1 (displays training progress)
- **Validation dataset:** `test_dataset`
- **Shuffle:** Enabled to randomize data order each epoch

```python
history = model.fit(train_dataset, epochs=10, verbose=1, validation_data=test_dataset, shuffle=True)


In [ ]:
history = model.fit(train_generator, epochs= 30, verbose= 1, validation_data= test_generator, shuffle= True)

# Training and Validation Loss & Accuracy

After training, you can visualize the model’s performance over epochs by plotting the loss and accuracy graphs for both training and validation datasets.

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plot training & validation loss values
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    average_precision_score,
    precision_recall_curve,
    matthews_corrcoef,
    balanced_accuracy_score,
    cohen_kappa_score
)

# Generating Predictions and Extracting True Labels

- The model predicts class probabilities for all samples in the test data generator.
- Predicted probabilities are converted into class indices using the argmax function.
- True class labels are retrieved from the test generator's `classes` attribute, ensuring correct correspondence with the predictions.


In [ ]:
pred=model.predict(test_generator)
y_pred=np.argmax(pred,axis=1)
y_test=test_generator.classes

# Confusion Matrix Visualization

- The confusion matrix is computed by comparing the true labels (`y_test`) with the predicted labels (`y_pred`).
- A heatmap is plotted with a large figure size for clarity.
- Each cell is annotated with the count of predictions.
- The color map used is `Blues` for intuitive visualization.
- The x-axis and y-axis are labeled with human-readable class names from the `classes` dictionary.
- X-axis labels are rotated 90 degrees and y-axis labels 45 degrees for better readability.
- A color bar is included to indicate the scale of values.

This visualization helps identify which classes are being correctly predicted and where misclassifications occur.


In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(30, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='winter', cbar=True)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.yticks(np.arange(len(classes)), classes.values(), rotation=0, fontsize=10)
plt.xticks(np.arange(len(classes)), classes.values(), rotation=90, fontsize=8)
plt.show()

# Classification Report

The classification report summarizes the main classification metrics for each class:

- **Precision:** How many selected items are relevant.
- **Recall:** How many relevant items are selected.
- **F1-score:** Harmonic mean of precision and recall.
- **Support:** Number of occurrences of each class in the true labels.

Make sure to pass your true labels (`y_test`) and predicted labels (`y_pred`) as inputs to the report.

This report helps evaluate the model’s performance across all classes and identify classes where the model performs well or poorly.


In [ ]:
report = classification_report(y_test, y_pred, target_names=list(classes.values()))
print(report)

# Visualizing Classification Metrics by Class

- This section generates a classification report to extract key performance metrics — **precision**, **recall**, and **F1-score** — for each class. The report is converted into a DataFrame, cleaned to exclude aggregate metrics, and then reshaped for easy plotting.

- The resulting bar plot displays these metrics side-by-side for each class, enabling quick comparison of the model’s performance across all categories. The y-axis is limited between 0 and 1 to represent score ranges, and class labels are rotated for readability.

- This visualization helps identify strengths and weaknesses per class at a glance.


In [ ]:
report_dict = classification_report(y_test, y_pred, target_names=list(classes.values()), output_dict=True)

df_report = pd.DataFrame(report_dict).transpose()
df_report = df_report.iloc[:-3, :]

# Reshape for seaborn
df_melt = df_report.reset_index().melt(id_vars='index', value_vars=['precision', 'recall', 'f1-score'],
                                      var_name='Metric', value_name='Score')
df_melt.rename(columns={'index': 'Class'}, inplace=True)

plt.figure(figsize=(30, 5))
sns.barplot(data=df_melt, x='Class', y='Score', hue='Metric')
plt.title('Classification Report Metrics by Class')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.xticks(rotation=90)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# Heatmap of Classification Metrics

- This heatmap provides a visual overview of the **precision**, **recall**, and **F1-score** for each class. The color gradient helps quickly spot classes where the model excels or needs improvement. Annotated values give exact scores for easy reference.

- Class names are kept horizontal for better readability, and the color bar indicates the metric scale from low to high.


In [ ]:
metrics_df = df_report[['precision', 'recall', 'f1-score']]

plt.figure(figsize=(12, 20))  # Adjust size as needed
sns.heatmap(metrics_df, annot=True, cmap='YlGnBu', cbar=True, linewidths=0.5, linecolor='gray')

plt.title('Classification Report Heatmap')
plt.xlabel('Metrics')
plt.ylabel('Classes')
plt.yticks(rotation=0)  # Keep class names horizontal for readability
plt.show()

### Precision-Recall Curves by Class

- This plot shows the **precision-recall curve** for each class, illustrating the trade-off between precision and recall at various thresholds. The **average precision (AP)** score for each class is included in the legend, summarizing the area under the curve.

- Precision-recall curves are especially useful for imbalanced datasets, as they focus on the model’s ability to correctly identify positive cases without being overwhelmed by negatives.

- The legend is placed outside the plot for better readability when dealing with multiple classes.


In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize

pred_probs = model.predict(test_generator)
num_classes = len(classes)
y_test_bin = label_binarize(y_test, classes=range(num_classes))

plt.figure(figsize=(20, 10))

for i in range(num_classes):
    precision, recall, _ = precision_recall_curve(y_test_bin[:, i], pred_probs[:, i])
    ap_score = average_precision_score(y_test_bin[:, i], pred_probs[:, i])
    plt.plot(recall, precision, label=f'{classes[i]} (AP={ap_score:.2f})')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves for Each Class')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')
plt.tight_layout()
plt.show()


# ROC Curves by Class

- This plot presents the **Receiver Operating Characteristic (ROC) curves** for each class, showing the balance between the **true positive rate** (sensitivity) and **false positive rate** at different classification thresholds.

- The **Area Under the Curve (AUC)** value for each class, displayed in the legend, quantifies the overall ability of the model to distinguish between classes — higher is better, with 1 being perfect and 0.5 representing random guessing.

- The diagonal dashed line represents random chance performance.

- The legend is positioned outside the plot to accommodate multiple classes clearly.

- ROC curves provide a comprehensive view of model performance across all thresholds, especially valuable for multi-class classification.


In [ ]:
from sklearn.metrics import roc_curve, auc

num_classes = len(classes)
y_test_bin = label_binarize(y_test, classes=range(num_classes))

plt.figure(figsize=(20, 10))

for i in range(num_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], pred_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{classes[i]} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves for Each Class')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')
plt.tight_layout()
plt.show()


In [ ]:
# Get one batch of images and labels from test_generator
images, true_labels = next(test_generator)  # shape (batch_size, h, w, c)
true_label_indices = np.argmax(true_labels, axis=1)  # adjust if not one-hot

# Predict probabilities for this batch
pred_probs = model.predict(images)
pred_label_indices = np.argmax(pred_probs, axis=1)
confidences = np.max(pred_probs, axis=1)

num_images = 25  # 5x5 grid
fig, axes = plt.subplots(5, 5, figsize=(15, 15))
axes = axes.flatten()

for i in range(num_images):
    ax = axes[i]
    img = images[i]

    # Normalize pixel values if outside 0-1
    if img.max() > 1.0 or img.min() < 0.0:
        img = (img - img.min()) / (img.max() - img.min())

    # If grayscale with channel dim
    if img.shape[-1] == 1:
        ax.imshow(img.squeeze(), cmap='gray')
    else:
        ax.imshow(img)

    ax.axis('off')

    true_idx = true_label_indices[i]
    pred_idx = pred_label_indices[i]
    confidence = confidences[i]

    color = 'green' if pred_idx == true_idx else 'red'
    title = f"P: {classes[pred_idx]}\nT: {classes[true_idx]}\nConf: {confidence:.2f}"
    ax.set_title(title, color=color, fontsize=10)

plt.tight_layout()
plt.show()
